In [ ]:
!pip install torch numpy scikit-learn hmmlearn -q

In [ ]:
import numpy as np
from pathlib import Path
DATASET_PATH = next(Path('/kaggle/input').rglob('sits_dataset.npz'))
data = np.load(DATASET_PATH, allow_pickle=True)
print(DATASET_PATH)
print(data['features'].shape)

In [ ]:
import json
features = data['features']
mask = data['attention_mask'].astype(bool)
flat = features[mask]
std = flat.std(axis=0)
std[std < 1e-6] = 1.0
stats = {'mean': flat.mean(axis=0).tolist(), 'std': std.tolist()}
with open('/kaggle/working/normalization_stats.json', 'w') as f:
    json.dump(stats, f, indent=2)
print('/kaggle/working/normalization_stats.json')

In [ ]:
import sys
from pathlib import Path
CODE_ROOT = next(Path('/kaggle/input').rglob('sits_bert/__init__.py')).parents[1]
sys.path.insert(0, str(CODE_ROOT))
from sits_bert.config import config
from sits_bert.pretrain import pretrain
config.dataset_path = str(DATASET_PATH)
config.normalization_stats_path = '/kaggle/working/normalization_stats.json'
config.output_dir = '/kaggle/working'
losses = pretrain(config)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(losses)
plt.title('Pretraining loss')
plt.xlabel('step')
plt.ylabel('loss')
plt.show()

In [ ]:
import os
[print(f, os.path.getsize(f'/kaggle/working/{f}')/1e6, 'MB') for f in os.listdir('/kaggle/working/')]

In [ ]:
print('=== PRETRAIN COMPLETE. Download: sits_bert_pretrained.pt + normalization_stats.json ===')